# Composable Option Strategy Builder

`OptionStrategy` lets you assemble multi-leg option strategies leg by leg
(`buy_call`, `sell_call`, `buy_put`, `sell_put`, `buy_underlying`,
`sell_underlying`), or reach for predefined constructors like spreads,
straddles, strangles, condors, and butterflies.

> **Scope note:** this is a static expiration-analysis tool. It does not
> model early exercise, margin, assignment, funding, slippage, taxes, or
> dynamic hedging.

**Sections:**
1. Build a strategy leg by leg
2. Compare predefined strategy constructors
3. Payoff profile table
4. Visualizations


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
from abaquant.derivatives import OptionStrategy
from abaquant.visualization import VisualizationError

## 1. Build a strategy leg by leg

A debit call spread: buy the 110 call, sell the 120 call.


In [ ]:
spread = OptionStrategy().buy_call(strike=110.0, premium=4.2).sell_call(strike=120.0, premium=1.8)

diagnostics = {
    "profit_at_125": spread.payoff(spot_price=125.0),
    "gross_payoff_at_125": spread.payoff(spot_price=125.0, include_premium=False),
    "net_inception_cost": spread.net_inception_cost(),
    "max_profit": spread.max_profit(),
    "max_loss": spread.max_loss(),
    "break_even_points": spread.break_even_points(),
}
for key, value in diagnostics.items():
    print(f"{key:22s}: {value}")

## 2. Compare predefined strategy constructors

Bull call spread, protective put, straddle, strangle, iron condor, and
butterfly — each built from one class-method call.


In [ ]:
strategies = {
    "bull_call_spread": OptionStrategy.bull_call_spread(
        lower_strike=110.0, upper_strike=120.0, lower_premium=4.2, upper_premium=1.8
    ),
    "protective_put": OptionStrategy.protective_put(
        underlying_entry_price=100.0, put_strike=95.0, put_premium=3.0
    ),
    "straddle": OptionStrategy.straddle(strike=100.0, call_premium=5.0, put_premium=4.5),
    "strangle": OptionStrategy.strangle(
        put_strike=95.0, call_strike=105.0, put_premium=3.0, call_premium=3.4
    ),
    "iron_condor": OptionStrategy.iron_condor(
        lower_put_strike=85.0, short_put_strike=95.0, short_call_strike=105.0,
        upper_call_strike=115.0, lower_put_premium=1.0, short_put_premium=3.2,
        short_call_premium=3.0, upper_call_premium=1.1,
    ),
    "butterfly": OptionStrategy.butterfly(
        lower_strike=90.0, middle_strike=100.0, upper_strike=110.0,
        lower_premium=12.0, middle_premium=6.0, upper_premium=2.0,
    ),
}

comparison = {
    name: {
        "legs": len(strategy.legs),
        "max_profit": strategy.max_profit(),
        "max_loss": strategy.max_loss(),
        "break_even_points": strategy.break_even_points(),
    }
    for name, strategy in strategies.items()
}
import pandas as pd
pd.DataFrame(comparison).T

## 3. Payoff profile table

Evaluate the bull call spread across a handful of terminal spot prices.


In [ ]:
profile = spread.profile(spot_prices=[90.0, 110.0, 112.4, 120.0, 130.0])
profile[["spot_price", "gross_payoff", "net_profit"]]

## 4. Visualizations

Aggregate payoff, per-leg components, and the iron condor payoff.


In [ ]:
try:
    figures = {
        "payoff": spread.visualize(chart="payoff"),
        "components": spread.visualize(chart="components"),
        "iron_condor": strategies["iron_condor"].visualize(chart="payoff"),
    }
    print(f"Created {len(figures)} figures: {list(figures)}")
except VisualizationError as exc:
    print(f"Visualization skipped (optional dependency missing): {exc}")

## Takeaway

Predefined constructors cover the common named strategies; `add_leg()`-style
chaining handles anything custom. Combine `.profile()` with `.visualize(chart="components")`
to see how each leg contributes to the overall payoff shape.
